# TP 05 : Analyse en Composantes Principales (ACP) — Métabolomique
**Master 2 Biochimie Appliquée — Université M'Hamed Bougara de Boumerdès (UMBB)**  
*Enseignante : Dr. Sarra BENMOUMOU (Ph.D.)*

---

## Objectifs :
1. Standardiser une matrice de biomarqueurs sériques.
2. Évaluer les valeurs propres et le pourcentage de variance expliquée.
3. Tracer le cercle des corrélations des métabolites.
4. Représenter la projection des patients dans le plan factoriel.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

df_meta = pd.read_csv('../datasets/metabolomique_serum.csv')
display(df_meta.head())


In [ ]:
metabolites = ['Glucose_mM', 'Triglycerides_gL', 'ALAT_UIL', 'ASAT_UIL', 'HDL_gL', 'Lactate_mM', 'BCAA_uM']
X = df_meta[metabolites].values
X_scaled = StandardScaler().fit_transform(X)

pca = PCA()
X_pca = pca.fit_transform(X_scaled)
var_exp = pca.explained_variance_ratio_ * 100

print(f"Variance Axe 1 : {var_exp[0]:.2f}%")
print(f"Variance Axe 2 : {var_exp[1]:.2f}%")


In [ ]:
# Cercle des corrélations
loadings = pca.components_[:2, :].T * np.sqrt(pca.explained_variance_[:2])

plt.figure(figsize=(6, 6))
circle = plt.Circle((0,0), 1, color='#1A365D', fill=False, linestyle='--')
plt.gca().add_patch(circle)

for i, var_name in enumerate(metabolites):
    plt.arrow(0, 0, loadings[i, 0], loadings[i, 1], head_width=0.03, color='#0D9488', length_includes_head=True)
    plt.text(loadings[i, 0]*1.1, loadings[i, 1]*1.1, var_name, fontweight='bold', color='#1A365D')

plt.xlim(-1.2, 1.2)
plt.ylim(-1.2, 1.2)
plt.axhline(0, color='gray', linestyle=':')
plt.axvline(0, color='gray', linestyle=':')
plt.title(f"Cercle des Corrélations (Axe 1: {var_exp[0]:.1f}% vs Axe 2: {var_exp[1]:.1f}%)", fontweight='bold')
plt.gca().set_aspect('equal', adjustable='box')
plt.show()


In [ ]:
# Plan des individus
df_pca = pd.DataFrame(X_pca[:, :2], columns=['PC1', 'PC2'])
df_pca['Statut'] = df_meta['Statut']

plt.figure(figsize=(8, 5))
sns.scatterplot(
    data=df_pca, x='PC1', y='PC2', hue='Statut', style='Statut', s=85,
    palette={'Temoin_Sain': '#2E7D32', 'Diabete_T2': '#1A365D', 'NASH_Hepatique': '#D97706'}
)
plt.title("Plan Factoriel des Patients (Discrimination Métabolique)", fontweight='bold')
plt.xlabel(f"Axe 1 ({var_exp[0]:.1f}%)")
plt.ylabel(f"Axe 2 ({var_exp[1]:.1f}%)")
plt.legend()
plt.show()
